# Notebook 02 — Preprocessing
**DNSC 6330 Capstone | HMDA LAR 2024**

Goals:
- Handle HMDA-specific null encodings
- Parse the DTI ratio range strings
- Encode categoricals
- Produce a clean, reproducible train/test split
- Preserve protected columns **separately** from feature matrix X

> **Key principle:** Protected attributes (`derived_race`, `derived_sex`, `derived_ethnicity`) must NOT enter the feature matrix X. They are preserved as a separate object `prot_train`/`prot_test` for post-hoc fairness analysis.

## 0. Imports

In [43]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
from sklearn.model_selection import train_test_split
warnings.filterwarnings('ignore')

import os
import json

# ── Same path block as Notebook 01 — must be in every notebook ───────────────
BASE_DIR    = os.getcwd()
FIGURES_DIR = os.path.join(BASE_DIR, 'figures')
TABLES_DIR  = os.path.join(BASE_DIR, 'tables')

os.makedirs(FIGURES_DIR, exist_ok=True)
os.makedirs(TABLES_DIR,  exist_ok=True)

pd.set_option('display.max_columns', 50)
plt.rcParams['figure.dpi'] = 120

print('Imports OK')
print(f'BASE_DIR: {BASE_DIR}')

Imports OK
BASE_DIR: /Users/tsotnedzeria-personal/Desktop/RML/capstone


## 1. Load Filtered Data from Notebook 01

In [44]:
df = pd.read_csv(os.path.join(BASE_DIR, 'df_filtered.csv'), low_memory=False)
print(f'Loaded: {df.shape}')
print(f'Label distribution:\n{df["label"].value_counts()}')
print(f'Approval rate: {df["label"].mean():.2%}')

Loaded: (8661748, 23)
Label distribution:
label
1    6558285
0    2103463
Name: count, dtype: int64
Approval rate: 75.72%


## 2. Handle HMDA-Specific Null Encodings

In [45]:
# HMDA uses 'Exempt', special numeric codes, and blanks to encode missing
HMDA_NA_VALUES = ['Exempt', 'NA', '', ' ', 'N/A', 'nan']

df_clean = df.replace(HMDA_NA_VALUES, np.nan).copy()

# HMDA numeric code for missing/not applicable varies by field
# Common exempt codes: 1111, 1111111, etc.
# For income: values > 9999 are often sentinel codes
INCOME_MAX = 9999  # HMDA income is reported in $000s; >9999 = likely sentinel
if 'income' in df_clean.columns:
    income_numeric = pd.to_numeric(df_clean['income'], errors='coerce')
    sentinel_mask = income_numeric > INCOME_MAX
    print(f'Income sentinel values (>{INCOME_MAX}): {sentinel_mask.sum():,}')
    df_clean.loc[sentinel_mask, 'income'] = np.nan

print(f'Shape after NA encoding: {df_clean.shape}')

Income sentinel values (>9999): 1,212
Shape after NA encoding: (8661748, 23)


In [46]:
# Drop rows missing label-critical features
REQUIRED_COLS = ['loan_amount', 'label']
REQUIRED_COLS = [c for c in REQUIRED_COLS if c in df_clean.columns]

before = len(df_clean)
df_clean = df_clean.dropna(subset=REQUIRED_COLS)
after = len(df_clean)
print(f'Dropped {before - after:,} rows missing required columns')
print(f'Remaining: {after:,}')

Dropped 0 rows missing required columns
Remaining: 8,661,748


## 3. Parse Debt-to-Income Ratio

HMDA reports DTI as range strings like `"36%-<50%"`. We convert to numeric midpoints.

In [47]:
def parse_dti(val):
    """
    Convert HMDA DTI range string to numeric midpoint.
    Examples:
      '36%-<50%'  → 43.0
      '<20%'      → 10.0
      '>60%'      → 65.0
      '50%'       → 50.0
      'Exempt'    → NaN
    """
    if pd.isna(val):
        return np.nan
    val = str(val).strip().replace('%', '').replace(' ', '')
    if val in ['Exempt', 'NA', 'nan', '']:
        return np.nan
    # Range: '36-<50'
    if '-' in val:
        parts = val.replace('<', '').replace('>', '').split('-')
        try:
            return (float(parts[0]) + float(parts[1])) / 2
        except (ValueError, IndexError):
            return np.nan
    # Less than: '<20'
    if val.startswith('<'):
        try:
            return float(val[1:]) / 2
        except ValueError:
            return np.nan
    # Greater than: '>60'
    if val.startswith('>'):
        try:
            return float(val[1:]) + 5
        except ValueError:
            return np.nan
    # Plain numeric
    try:
        return float(val)
    except ValueError:
        return np.nan


if 'debt_to_income_ratio' in df_clean.columns:
    # Show unique values before parsing
    print('DTI unique values (sample):', df_clean['debt_to_income_ratio'].unique()[:15])
    df_clean['debt_to_income_ratio'] = df_clean['debt_to_income_ratio'].apply(parse_dti)
    print('\nAfter parsing:')
    print(df_clean['debt_to_income_ratio'].describe())

DTI unique values (sample): [nan '50%-60%' '37' '46' '45' '48' '30%-<36%' '49' '41' '38' '44' '43'
 '20%-<30%' '39' '40']

After parsing:
count    7.719834e+06
mean     4.013858e+01
std      1.321845e+01
min      1.000000e+01
25%      3.300000e+01
50%      4.100000e+01
75%      4.800000e+01
max      6.500000e+01
Name: debt_to_income_ratio, dtype: float64


## 4. Feature Selection & Type Coercion

In [48]:
# Core predictive features — non-protected
FEATURE_COLS = [
    'loan_amount', 'income', 'debt_to_income_ratio', 'loan_to_value_ratio',
    'property_value', 'loan_term',       # numeric
    'loan_type', 'loan_purpose', 'occupancy_type', 'lien_status',
    'conforming_loan_limit', 'preapproval',
    'submission_of_application',
    'applicant_credit_score_type', 'co_applicant_credit_score_type',  # categorical
]
# Protected class features — kept SEPARATE from X, never trained on
PROTECTED_COLS = [
    'derived_race',
    'derived_sex',
    'derived_ethnicity',
    'applicant_age',
    'co_applicant_age'
]

# Filter to columns that actually exist in the dataset
FEATURE_COLS   = [c for c in FEATURE_COLS   if c in df_clean.columns]
PROTECTED_COLS = [c for c in PROTECTED_COLS if c in df_clean.columns]

print(f'Feature columns ({len(FEATURE_COLS)}):   {FEATURE_COLS}')
print(f'Protected columns ({len(PROTECTED_COLS)}): {PROTECTED_COLS}')

# Subset to model columns
df_model = df_clean[FEATURE_COLS + PROTECTED_COLS + ['label']].copy()

Feature columns (14):   ['loan_amount', 'income', 'debt_to_income_ratio', 'property_value', 'loan_term', 'loan_type', 'loan_purpose', 'occupancy_type', 'lien_status', 'conforming_loan_limit', 'preapproval', 'submission_of_application', 'applicant_credit_score_type', 'co_applicant_credit_score_type']
Protected columns (5): ['derived_race', 'derived_sex', 'derived_ethnicity', 'applicant_age', 'co_applicant_age']


In [49]:
# Coerce numeric columns
NUMERIC_COLS = [
    'loan_amount', 'income', 'property_value',
    'debt_to_income_ratio', 'loan_to_value_ratio', 'loan_term',
]
NUMERIC_COLS = [c for c in NUMERIC_COLS if c in df_model.columns]

for col in NUMERIC_COLS:
    df_model[col] = pd.to_numeric(df_model[col], errors='coerce')

print('Dtypes after coercion:')
print(df_model.dtypes)

Dtypes after coercion:
loan_amount                         int64
income                            float64
debt_to_income_ratio              float64
property_value                    float64
loan_term                         float64
loan_type                           int64
loan_purpose                        int64
occupancy_type                      int64
lien_status                         int64
conforming_loan_limit              object
preapproval                         int64
submission_of_application           int64
applicant_credit_score_type         int64
co_applicant_credit_score_type      int64
derived_race                       object
derived_sex                        object
derived_ethnicity                  object
applicant_age                      object
co_applicant_age                   object
label                               int64
dtype: object


## 5. Encode Categorical Features

In [50]:
# Collapse credit score type to binary
# applicant_credit_score_type encodes which scoring MODEL was used (FICO, Equifax etc.)
# not the actual score — not a meaningful predictive feature as multi-category
# Binary: 1 = score exists, 0 = not available/exempt/not applicable
# Codes 1 and 1111 = no score available

df_model['has_credit_score'] = (
    df_model['applicant_credit_score_type']
    .apply(lambda x: 0 if str(x).strip() in ['9', '1111'] else 1)
)
df_model['co_applicant_has_credit_score'] = (
    df_model['co_applicant_credit_score_type']
    .apply(lambda x: 0 if str(x).strip() in ['9', '10', '1111'] else 1)
)
# Drop originals
df_model = df_model.drop(
    columns=['applicant_credit_score_type', 'co_applicant_credit_score_type'],
    errors='ignore'
)

print('Credit score type collapsed to binary.')
print(df_model['has_credit_score'].value_counts())
print(df_model['co_applicant_has_credit_score'].value_counts())

Credit score type collapsed to binary.
has_credit_score
1    7362538
0    1299210
Name: count, dtype: int64
co_applicant_has_credit_score
0    6946570
1    1715178
Name: count, dtype: int64


In [51]:
CAT_COLS = [
    'loan_type', 'loan_purpose', 'occupancy_type', 'lien_status',
    'conforming_loan_limit', 'preapproval',
    'submission_of_application',
    'applicant_credit_score_type', 'co_applicant_credit_score_type',]
CAT_COLS = [c for c in CAT_COLS if c in df_model.columns]

print(f'Encoding {len(CAT_COLS)} categorical columns: {CAT_COLS}')
print('Cardinality before encoding:')
for c in CAT_COLS:
    print(f'  {c}: {df_model[c].nunique()} unique values')

df_model = pd.get_dummies(df_model, columns=CAT_COLS, drop_first=True)

print(f'\nShape after encoding: {df_model.shape}')

Encoding 7 categorical columns: ['loan_type', 'loan_purpose', 'occupancy_type', 'lien_status', 'conforming_loan_limit', 'preapproval', 'submission_of_application']
Cardinality before encoding:
  loan_type: 4 unique values
  loan_purpose: 6 unique values
  occupancy_type: 3 unique values
  lien_status: 2 unique values
  conforming_loan_limit: 3 unique values
  preapproval: 2 unique values
  submission_of_application: 3 unique values

Shape after encoding: (8661748, 29)


## 6. Final Feature Matrix Construction

Separate X (features), y (label), and protected columns before the train/test split.

In [52]:
# Build final feature set — exclude label and protected cols from X
EXCLUDE_FROM_X = ['label'] + PROTECTED_COLS
feature_final  = [c for c in df_model.columns if c not in EXCLUDE_FROM_X]

X         = df_model[feature_final].copy()
y         = df_model['label'].copy()
protected = df_model[PROTECTED_COLS].copy()

print(f'X shape:         {X.shape}')
print(f'y shape:         {y.shape}')
print(f'protected shape: {protected.shape}')
print(f'Label balance:   {y.mean():.2%} approved')
print(f'\nFeature columns:')
print(feature_final)

X shape:         (8661748, 23)
y shape:         (8661748,)
protected shape: (8661748, 5)
Label balance:   75.72% approved

Feature columns:
['loan_amount', 'income', 'debt_to_income_ratio', 'property_value', 'loan_term', 'has_credit_score', 'co_applicant_has_credit_score', 'loan_type_2', 'loan_type_3', 'loan_type_4', 'loan_purpose_2', 'loan_purpose_4', 'loan_purpose_5', 'loan_purpose_31', 'loan_purpose_32', 'occupancy_type_2', 'occupancy_type_3', 'lien_status_2', 'conforming_loan_limit_NC', 'conforming_loan_limit_U', 'preapproval_2', 'submission_of_application_2', 'submission_of_application_1111']


In [53]:
# Missingness summary for final X
miss_X = X.isnull().mean().sort_values(ascending=False)
miss_X = miss_X[miss_X > 0]
print('── Missing values in X (pre-imputation) ──')
if miss_X.empty:
    print('  No missing values.')
else:
    print(miss_X.to_string())

── Missing values in X (pre-imputation) ──
debt_to_income_ratio    0.108744
property_value          0.065778
income                  0.063118
loan_term               0.047581


## 7. Train / Test Split

Stratified split preserves class balance across train and test sets.

In [54]:
X_train, X_test, y_train, y_test, prot_train, prot_test = train_test_split(
    X, y, protected,
    test_size=0.20,
    random_state=42,
    stratify=y        # preserve class balance
)

print(f'Train: {X_train.shape[0]:>8,} rows  |  approval rate: {y_train.mean():.2%}')
print(f'Test:  {X_test.shape[0]:>8,} rows  |  approval rate: {y_test.mean():.2%}')
print()

# Sanity check: protected cols preserved correctly
assert len(prot_train) == len(X_train), 'Protected train size mismatch'
assert len(prot_test)  == len(X_test),  'Protected test size mismatch'
print('Sanity checks passed.')

Train: 6,929,398 rows  |  approval rate: 75.72%
Test:  1,732,350 rows  |  approval rate: 75.72%

Sanity checks passed.


## 8. Imputation

Median imputation fitted on **train only** to avoid data leakage.

In [55]:
# Fit imputation on train, apply to both train and test
train_medians = X_train.median(numeric_only=True)

X_train_imputed = X_train.fillna(train_medians)
X_test_imputed  = X_test.fillna(train_medians)

# Verify no remaining nulls
assert X_train_imputed.isnull().sum().sum() == 0, 'NaNs remain in X_train after imputation'
assert X_test_imputed.isnull().sum().sum()  == 0, 'NaNs remain in X_test after imputation'

print('Imputation complete — no remaining NaNs.')
print('\nImputed medians (train):')
print(train_medians[train_medians.notna()].to_string())

Imputation complete — no remaining NaNs.

Imputed medians (train):
loan_amount                       205000.0
income                               104.0
debt_to_income_ratio                  41.0
property_value                    385000.0
loan_term                            360.0
has_credit_score                       1.0
co_applicant_has_credit_score          0.0
loan_type_2                            0.0
loan_type_3                            0.0
loan_type_4                            0.0
loan_purpose_2                         0.0
loan_purpose_4                         0.0
loan_purpose_5                         0.0
loan_purpose_31                        0.0
loan_purpose_32                        0.0
occupancy_type_2                       0.0
occupancy_type_3                       0.0
lien_status_2                          0.0
conforming_loan_limit_NC               0.0
conforming_loan_limit_U                0.0
preapproval_2                          1.0
submission_of_application_2   

## 9. Save Artifacts

In [56]:
import os
import json

# ── Paths — same pattern as Notebook 01 ─────────────────────────────────────
BASE_DIR    = os.getcwd()
FIGURES_DIR = os.path.join(BASE_DIR, 'figures')
TABLES_DIR  = os.path.join(BASE_DIR, 'tables')
os.makedirs(FIGURES_DIR, exist_ok=True)
os.makedirs(TABLES_DIR,  exist_ok=True)

# ── Save all artifacts to the same folder as the notebooks ───────────────────

# Raw (non-imputed) splits — needed by notebooks that operate on original NaN structure
X_train.to_parquet(os.path.join(BASE_DIR, 'X_train.parquet'))
X_test.to_parquet(os.path.join(BASE_DIR, 'X_test.parquet'))
y_train.to_frame().to_parquet(os.path.join(BASE_DIR, 'y_train.parquet'))
y_test.to_frame().to_parquet(os.path.join(BASE_DIR, 'y_test.parquet'))
prot_train.to_parquet(os.path.join(BASE_DIR, 'prot_train.parquet'))
prot_test.to_parquet(os.path.join(BASE_DIR, 'prot_test.parquet'))

# Imputed versions for direct model input
X_train_imputed.to_parquet(os.path.join(BASE_DIR, 'X_train_imputed.parquet'))
X_test_imputed.to_parquet(os.path.join(BASE_DIR, 'X_test_imputed.parquet'))

# Median values used for imputation — needed for reproducibility and deployment
train_medians.to_frame('median').to_parquet(os.path.join(BASE_DIR, 'train_medians.parquet'))

print('Saved artifacts:')
for f in ['X_train', 'X_test', 'y_train', 'y_test', 'prot_train', 'prot_test',
          'X_train_imputed', 'X_test_imputed', 'train_medians']:
    path = os.path.join(BASE_DIR, f'{f}.parquet')
    size = os.path.getsize(path) / 1e6
    print(f'  {f}.parquet  ({size:.1f} MB)')

Saved artifacts:
  X_train.parquet  (87.3 MB)
  X_test.parquet  (21.9 MB)
  y_train.parquet  (38.7 MB)
  y_test.parquet  (9.8 MB)
  prot_train.parquet  (50.3 MB)
  prot_test.parquet  (12.6 MB)
  X_train_imputed.parquet  (86.4 MB)
  X_test_imputed.parquet  (21.6 MB)
  train_medians.parquet  (0.0 MB)


In [57]:
# Final preprocessing summary
print('══ Preprocessing Summary ══')
print(f'  Raw rows:                {len(df_clean):>10,}')
print(f'  Final rows (post-clean): {len(X):>10,}')
print(f'  Features in X:           {X.shape[1]:>10,}')
print(f'  Protected cols kept:     {PROTECTED_COLS}')
print(f'  Train size:              {len(X_train):>10,}')
print(f'  Test size:               {len(X_test):>10,}')
print(f'  Imputation:              median (fitted on train only)')
print(f'  Random seed:             42')

══ Preprocessing Summary ══
  Raw rows:                 8,661,748
  Final rows (post-clean):  8,661,748
  Features in X:                   23
  Protected cols kept:     ['derived_race', 'derived_sex', 'derived_ethnicity', 'applicant_age', 'co_applicant_age']
  Train size:               6,929,398
  Test size:                1,732,350
  Imputation:              median (fitted on train only)
  Random seed:             42
